# Large Language Model (Zero-Shot) for Information Extraction

## Overview

This notebook explores a **zero-shot information extraction approach using Large Language Models (LLMs)** for extracting structured PIO elements—**Population, Intervention, and Outcome** from clinical trial abstracts in the **EBM-NLP dataset**.

Unlike rule-based and supervised models, the zero-shot approach does not require task-specific training data. Instead, it leverages the inherent reasoning and language understanding capabilities of pre-trained LLMs to directly identify relevant entities from unstructured biomedical text based on carefully designed prompts.

The pipeline involves:

- Designing structured prompts to guide the model in identifying PIO components  
- Feeding clinical trial abstracts directly into the LLM without fine-tuning  
- Extracting and formatting model outputs into a structured table schema  


In [ ]:
import os
import csv
import json
import sys
import ast
import warnings
warnings.filterwarnings("ignore")

In [ ]:
!pip install -U bitsandbytes transformers accelerate

In [ ]:
!pip install importnb

In [ ]:
import sys
from importnb import Notebook

# Tell Python where the folder is (change this to your actual folder path)
sys.path.append("/content/drive/MyDrive/Colab Notebooks/Zero Shot/")

# Now import it like a normal library
with Notebook():
    import Utils

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from google.colab import userdata

# Load the model and tokenizer
model_id = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit" # This specific version is pre-quantized for speed

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    dtype=torch.bfloat16,
    device_map="auto") # This puts it on the GPU automatically

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
df = pd.read_excel(r"/content/drive/MyDrive/Colab Notebooks/cwk_data/cwk_train_data.xlsx")

In [ ]:
df.head(2)

In [ ]:
#Zero Shot

In [ ]:
def extract_zero_shot(document_text):
    # This prompt defines exactly what goes into P, I, and O based on your schema
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a strict data extraction tool.
**RULES**:
1. ONLY extract words that are DIRECTLY written in the provided text.
2. If an entity is not explicitly mentioned, return [].
3. DO NOT use external knowledge.
4. DO NOT provide any introductory text or notes.
5. Output MUST be valid JSON only.
6. Participants : Extract Age, Sex, Sample Size, Disease and Condition given in the text.
7. Intervention : Extract name of the drugs which are given in the text, dont manipulate the name of the drugs .
8. Outcome : Extract the outcomes of the medical trial from the text.

### SCHEMA:
{{
  "Participants": [],
  "Intervention": [],
  "Outcome": []
}}

Text: {document_text}<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # We use temperature 0 for consistent, reproducible results (important for your project)
    outputs = model.generate(
        **inputs,
        max_new_tokens=256,
        temperature=0.0,
        do_sample=False
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    json_string = response.split("assistant")[-1].strip()
    try:
        return json.loads(json_string)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        print(f"Malformed JSON string: {json_string}")
        return {}


In [ ]:
file_path = r"/content/drive/MyDrive/Colab Notebooks/Zero Shot/result_zero_shot_2.csv"

In [ ]:
Utils.config(df, file_path, 200, extract_zero_shot )